# Fantasy Football Projections - Quick Start

This notebook demonstrates the end-to-end workflow:
1. Load and prepare data
2. Build features
3. Train GBM model
4. Evaluate and visualize results

In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from ffproj.data import load_weekly_stats
from ffproj.features import build_all_features, select_feature_columns
from ffproj.models_gbm import GBMQuantileEnsemble
from ffproj.metrics import compute_all_metrics, plot_calibration_curve
from ffproj.utils import set_random_seed

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Set seed for reproducibility
set_random_seed(42)

## 1. Load Data

In [ ]:
# Load weekly stats for 2019-2023
seasons = [2019, 2020, 2021, 2022, 2023]
df = load_weekly_stats(seasons, cache=True)

print(f"Loaded {len(df)} player-weeks")
print(f"\nPositions: {df['position'].value_counts()}")
print(f"\nSeasons: {df['season'].value_counts().sort_index()}")

In [ ]:
# Quick EDA
df[['fp_ppr', 'fp_half_ppr']].describe()

## 2. Feature Engineering

In [ ]:
# Sort by player and time
df = df.sort_values(['player_id', 'season', 'week']).reset_index(drop=True)

# Build features
df = build_all_features(df)

# Select feature columns
feature_cols = select_feature_columns(df)
print(f"Created {len(feature_cols)} features")
print(f"\nExample features:\n{feature_cols[:10]}")

## 3. Train/Val Split

In [ ]:
# Use 2019-2022 for training, 2023 for validation
train_df = df[df['season'].isin([2019, 2020, 2021, 2022])]
val_df = df[df['season'] == 2023]

# Remove rows with missing target
train_df = train_df.dropna(subset=['fp_ppr'])
val_df = val_df.dropna(subset=['fp_ppr'])

print(f"Training samples: {len(train_df)}")
print(f"Validation samples: {len(val_df)}")

## 4. Train GBM Model

In [ ]:
# Prepare data
X_train = train_df[feature_cols].fillna(0)
y_train = train_df['fp_ppr']
X_val = val_df[feature_cols].fillna(0)
y_val = val_df['fp_ppr']

# Train model
model = GBMQuantileEnsemble(verbose=True)
history = model.train(X_train, y_train, X_val, y_val)

## 5. Evaluate Model

In [ ]:
# Make predictions
preds = model.predict(X_val)

# Compute metrics
from ffproj.metrics import mae, rmse, coverage_at

metrics = {
    'MAE': mae(y_val, preds['q50']),
    'RMSE': rmse(y_val, preds['q50']),
    'Coverage_80': coverage_at(y_val, preds['q10'], preds['q90']),
}

print("\nValidation Metrics:")
for name, value in metrics.items():
    print(f"{name}: {value:.4f}")

In [ ]:
# Predicted vs Actual
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter plot
axes[0].scatter(preds['q50'], y_val, alpha=0.3)
axes[0].plot([0, y_val.max()], [0, y_val.max()], 'r--', label='Perfect')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')
axes[0].set_title('Predicted vs Actual Fantasy Points')
axes[0].legend()

# Residuals
residuals = y_val - preds['q50']
axes[1].hist(residuals, bins=50, alpha=0.7)
axes[1].axvline(0, color='r', linestyle='--')
axes[1].set_xlabel('Residual (Actual - Predicted)')
axes[1].set_ylabel('Count')
axes[1].set_title('Residual Distribution')

plt.tight_layout()
plt.show()

In [ ]:
# Feature importance
importance = model.get_feature_importance()

plt.figure(figsize=(10, 8))
plt.barh(importance.head(20)['feature'], importance.head(20)['importance'])
plt.xlabel('Importance')
plt.title('Top 20 Feature Importance')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# Calibration curve
fig = plot_calibration_curve(
    y_val.values,
    preds['q10'],
    preds['q90'],
    n_bins=10
)
plt.show()

## 6. Evaluate by Position

In [ ]:
# Metrics by position
results_by_pos = []

for pos in val_df['position'].unique():
    pos_mask = val_df['position'] == pos
    pos_metrics = {
        'Position': pos,
        'Count': pos_mask.sum(),
        'MAE': mae(y_val[pos_mask], preds['q50'][pos_mask]),
        'RMSE': rmse(y_val[pos_mask], preds['q50'][pos_mask]),
        'Coverage': coverage_at(
            y_val[pos_mask],
            preds['q10'][pos_mask],
            preds['q90'][pos_mask]
        )
    }
    results_by_pos.append(pos_metrics)

results_df = pd.DataFrame(results_by_pos)
print("\nMetrics by Position:")
print(results_df.to_string(index=False))

## 7. Example: Top 20 Projections for Week 1

In [ ]:
# Get week 1 predictions
week1 = val_df[val_df['week'] == 1].copy()
week1_X = week1[feature_cols].fillna(0)
week1_preds = model.predict(week1_X)

# Create results dataframe
week1_results = pd.DataFrame({
    'Player': week1['player_name'].values if 'player_name' in week1 else week1['player_id'].values,
    'Position': week1['position'].values,
    'P10': week1_preds['q10'],
    'Projection': week1_preds['q50'],
    'P90': week1_preds['q90'],
    'Actual': week1['fp_ppr'].values
})

# Sort by projection
week1_results = week1_results.sort_values('Projection', ascending=False)

print("\nTop 20 Projections for Week 1:")
print(week1_results.head(20).to_string(index=False))

## Next Steps

1. Run rolling backtest: `python -m ffproj.train --backtest`
2. Compare to baselines
3. Try deep learning models (TCN/LSTM)
4. Launch Streamlit app: `streamlit run src/app/streamlit_app.py`